# preloading

In [ ]:
import os
from pathlib import Path
import datetime
import time
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import functools
import itertools

#os.environ["PYLAB_DB_LOCAL"] = ""
#os.environ["PYLAB_DB_OUT"] = ""
import pyflexlab
from pyomnix.utils import CM_TO_INCH, next_lst_gen, constant_generator
from pyomnix import DataManipulator
from pyflexlab import FileOrganizer, DataProcess, MeasureFlow

from pyflexlab.measure_flow import MeasurementRecipe, PlotRecipe
from pyomnix.omnix_logger import get_logger
# Measure module will need the NIDAQmx module to work,
from pyflexlab.recipe_builders import (
    RecipeBuilder,
    RecipeOptions,
    MeasureModules,
    PlotSeries,
    PlotModules
)

In [2]:
project_name = "Date-Material"
folder = DataProcess(project_name)
measureflow = MeasureFlow(project_name)
plotobj = DataManipulator(2)
#Folder.add_measurement("RT")
#Folder.add_plan("RT","__whole, measure the whole RT")

In [ ]:
folder.tree()

### Add measurement type if needed

In [ ]:
#FileOrganizer.add_measurement_type("V_source_sweep_ac","Vmax{maxv}V-step{stepv}V-freq{freq}Hz-{vhigh}-{vlow}", overwrite=False)

In [ ]:
#FileOrganizer.name_fstr_gen("I_source_sweep_dc","V_sense","T_vary")

In [ ]:
#DataPlot.gui_pan_color()

# Setup and Parameter

In [ ]:
#measureflow.get_visa_resources()   # list all VISA resources
#measureflow.load_meter("6221","GPIB0::12::INSTR")
#measureflow.load_meter("2182","GPIB0::7::INSTR")
#measureflow.load_meter("2400","GPIB0::23::INSTR")
#measureflow.load_meter("6430","GPIB0::24::INSTR")
#measureflow.load_meter("sr830","GPIB0::8::INSTR","GPIB0::9::INSTR")
#measureflow.load_mercury_ips("TCPIP0::10.10.24.10::7020::SOCKET")
#measureflow.load_mercury_itc("TCPIP0::10.10.10.24::7020::SOCKET")
#measureflow.load_ITC503("GPIB0::23::INSTR","GPIB0::24::INSTR")
#measureflow.load_rotator()
measureflow.load_fakes(3)

*****
# Guidance for measurement

### *sweep modes:*
- DC
    - 0-max-0
    - 0--max-max-0
    - manual
- AC
    - 0-max-0
    - 0-max
    - manual

### Column Names:
- time (default in generator)
- X_source
- X (sense->ext) 
    - (if lock-in, then X, Y, R, Theta)

### Combination of Varies or Mappings
- set `if_combine_gen=False` in `get_mea_dict` to get the list of generators instead of a whole list generator
- use `constants.next_lst_gen` to get the next values list
- replace mapping generators with constant generators (careful about the values) to temporarily pause the mapping and do varying, and remember to vary circularly (hysteretically) use `reverse=True`

******

## Recipe-builder full flow test

This cell repeats the fake R-H loop test through the modular `recipe_builders` API. It uses the fake meters prepared above and does not load meters inside the recipe flow.


In [ ]:
measure_type = ""  # parent folder name for the measurement series
measure_comment = ""  # sub folder name to distinguish some special measurements (repeat or test, etc.)
ds_source = measureflow.instrs["fakes"][0]
gate_meter = measureflow.instrs["fakes"][1]

# HOOKs
def after_prepare_action(meadict: dict):
    # happens after measure generator is created, before any actual steps
    pass
def on_measure_action(records: dict):
    # happens right after retrieving data from meters
    pass
def on_record_action(records: dict):
    # happens after records have been written
    pass
def shutdown_actions():
    """
    happens when shutdown procedure is triggered
    """
    pass
# shutdown procedures
#::meter targets(call target.output_switch("off"))
#::callable without output_switch property(call target())
shutdown_targets = (shutdown_actions,)
#===========Extra Options==============
builder = RecipeBuilder(
    options=RecipeOptions(
        if_combine_gen=True,
        special_name=measure_comment,
        measure_nickname=measure_type,
        vary_loop=True,
        source_wait=0.3,
        wait_before_vary=5,
    )
)
#===========Plot Modules=======================
series = [
        PlotSeries(0, 0, 0, 0, 5, "t", "B"),
        PlotSeries(0, 1, 0, 5, 3, "B", "V"),
    ]
plot_recipe =  PlotModules.mapped_plot(
    init_args=(1, 2, 1),
    titles=[["t B", "B V"]],
    series=series,
    saving_interval=7,
    plotobj=plotobj,
    use_dash=True,
    init_kwargs=None,
    update_kwargs=None,
)
#=============Measure Modules==============
(builder
.add(MeasureModules.fixed_current_source(1, high=0, low=0, meter=ds_source, compliance=1))
.add(MeasureModules.fixed_voltage_source(2, high=0, low=0, meter=gate_meter, compliance=1))
.add(MeasureModules.voltage_sense(high=0, low=0, comment="", meter=ds_source, ac_dc="dc"))
.add(MeasureModules.current_sense(high=0, low=0, comment="", meter=gate_meter, ac_dc= "dc"))
.add(MeasureModules.vary_magnetic_field(start=-1, stop=1))
.add(MeasureModules.fixed_temperature(300)))
#===============Assemble=======================
builder_recipe = (
    builder.build(
        step_time=1,
        plot=plot_recipe,
        after_prepare=after_prepare_action,
        on_measure=on_measure_action,
        on_record=on_record_action,
        shutdown=shutdown_targets
    )
)

builder_result = measureflow.run_recipe(builder_recipe)

print("recipe-builder file:", builder_result["file_path"])
print("recipe-builder columns:", builder_result["record_num"])
print("recipe-builder modules:", builder_recipe.measure_mods)
